In [5]:
import matplotlib.pyplot as plt #플레이팅
import numpy as np # 숫자
import PIL # 사진을 열고 다루는 이미지도구
import tensorflow as tf # 최첨단 주방기계
from tensorflow.keras import layers # 주방에 필요한 조리장비
import os # 운영체재  폴더파일관리
import pathlib #파일경로 다루는 도구
import urllib.request # 배달 
import tarfile # 알집 압출풀기 

In [6]:
# 꽃사진 다운로드 하고 압출 풀기
def download_and_extract_data(): # 배달직원에게 업무지시서 작성
    dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz" #꽃사진 인터넷주소
    
    download_dir = os.path.join(os.getcwd(), 'flower_dateset') # 현재폴더안에 창고 경로 만들기
    os.makedirs(download_dir, exist_ok=True) # 창고 만들기, 기존에 있으면 무시
    
    tgz_path = os.path.join(download_dir, "flower_photos.tgz") # 다운로드할 압출 꾸러미의 저장경로 
    
    if not os.path.exists(tgz_path): #압출꾸러미 창고에 아직없으면
        print("데이터셋 다운로드중..") # 배달 시작알림
        urllib.request.urlretrieve(dataset_url, tgz_path) # 꽃사진 내려받아 저장
        print("데이터셋 다운로드 완료") # 배달 완료알림
        
        if not os.path.exists(os.path.join(download_dir, "flower_photos")): # 압축을 푼 사진폴더가 없으면
            print("압축풀기중..") # 압축풀기 시작알림
            with tarfile.open(tgz_path, "r:gz") as tar: # 압축 열기 
                tar.extractall(path=download_dir) # 압축풀기 
            print("압축풀기 완료") # 압축풀기 완료알림
            
        return os.path.join(download_dir, "flower_photos") # 꽃사진 최종 폴더 경로
        
        

In [ ]:
# 꽃사진 창고 열기

data_dir = download_and_extract_data() #배달 직원 호출  다운로드,압축 해제 실행

daita_dir = pathlib.Path(data_dir) # 창주소를 편리하게 관리할수있는 도구로 바꾸기 

데이터셋 다운로드중..
데이터셋 다운로드 완료
압축풀기중..


C:\Users\human-01\AppData\Local\Temp\ipykernel_3120\3699133611.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=download_dir) # 압축풀기


압축풀기 완료


In [17]:
# 꽃 종류와 전체 사진 개수 확인

# 문자열로 된 창고 주소를 경로 객체로 변환
data_dir = pathlib.Path(data_dir)  # pathlib 경로 도구로 바꿔서 glob() 사용 가능하게 만들기

print(data_dir)  # 창고 주소 확인
print(data_dir.exists())  # 창고가 실제로 존재하는지 확인

classes = sorted([item.name for item in data_dir.glob('*/') if item.is_dir()]) # 하위폴더 이름가져와 꽃종류별 로 정렬
image_count = len(list(data_dir.glob('*/*.jpg'))) # 모든 꽃 폴더안의 JPG 사진 개수 세기 

print(f"발견된 클래스: {classes}") # 꽃종류 출력
print(f"발견된 이미지 개수: {image_count}") # 꽃사진 개수 출력
print(data_dir)
print(classes)
print(data_dir.exists())

                 

c:\Users\human-01\OneDrive\Desktop\Python_AI_Lab\flower_dateset\flower_photos
True
발견된 클래스: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
발견된 이미지 개수: 3670
c:\Users\human-01\OneDrive\Desktop\Python_AI_Lab\flower_dateset\flower_photos
['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
True


In [18]:
# 꽃 종류별 사진 개수 확인

for class_name in classes: #꽃 종류를 하나씩 꺼내
    class_images = len(list(data_dir.glob(f'{class_name}/*'))) # 해당 꽃 폴더 안의 파일 개수세기
    print(f"{class_name}: {class_images} 개 이미지") # 꽃이름 ,사진개수 출력 

daisy: 633 개 이미지
dandelion: 898 개 이미지
roses: 641 개 이미지
sunflowers: 699 개 이미지
tulips: 799 개 이미지


In [19]:
# 이미지 크기와 배치크기설정
img_height =180 #높이
img_width = 180 #너비
batch_size = 32 # 한번에 꽃사진 32장씩묶어서 처리


In [23]:
#훈련 데이터셋 생성

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    str(data_dir), #꽃 사진 창고 주소 전달 
    validation_split=0.2, #20% 는 검증용 분리
    subset="training", # 학습용 80% 만 가져오기 
    seed=123, #랜덤으로 데이터 나눔 기준 설정 
    image_size=(img_height, img_width), # 꽃사진 크기 설정 
    batch_size=batch_size) # 한번에 꽃사진 32장씩 묶어서 처리



Found 3670 files belonging to 5 classes.
Using 2936 files for training.


In [24]:
# 검증 데이터셋 만들기
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    str(data_dir), #꽃 사진 창고 주소 전달 
    validation_split=0.2, #20% 는 검증용 분리
    subset="validation", # 검증용 20% 만 가져오기 
    seed=123, #랜덤으로 데이터 나눔 기준 설정 
    image_size=(img_height, img_width), # 꽃사진 크기 설정 
    batch_size=batch_size) # 한번에 꽃사진 32장씩 묶어서 처리


Found 3670 files belonging to 5 classes.
Using 734 files for validation.


In [25]:
# 꽃종류 확인하기
class_names= train_ds.class_names # 꽃 폴더 이름을 정답표 목록으로 가져오기
print("클래스:", class_names) # 꽃종류 출력

num_classes = len(class_names) # 꽃종류의 총개수를 세어모델출력층에 사용 

클래스: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [26]:
# 데이터 처리 속도 최적화

AUTOTUNE = tf.data.AUTOTUNE #최적화 설정
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)# 학습 캐시에 저장하고 순서를 섞고, 다음 데이터 미리준비

val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)# 검증 캐시저장ㅎ아고 다음 데이터 미리준비 
                                 